# **EduSense: Stage A — Feature Extraction**
**King Khalid University — Graduation Project 2025**

---

## Pipeline
```
DAISEE Videos → Read Frames → AffectNet-ViT (GPU batch) → Save .npy embeddings
```

**Key design decisions:**
- ✅ No YOLO — DAISEE videos are already face-centered
- ✅ Batch processing — all 30 frames sent to GPU at once per video
- ✅ AffectNet-ViT — pretrained on facial expressions (not ImageNet)
- ✅ Resume support — skips already-extracted videos
- ✅ Saves to Google Drive

**⚠️ Run this notebook ONCE.**

## 📦 **1. Install Dependencies**

In [ ]:
!pip install -q ultralytics transformers opencv-python-headless tqdm kagglehub Pillow
print('✅ Done')

## 📚 **2. Imports**

In [ ]:
import os
import json
import shutil
import numpy as np
import pandas as pd
import cv2
import torch
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
from transformers import ViTModel, ViTImageProcessor
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  WARNING: No GPU found! Extraction will be very slow.')

## 💾 **3. Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/daisee_embeddings_affectnet'
print(f'Embeddings will be saved to: {SAVE_DIR}')

## 📥 **4. Download DAISEE Dataset**

In [ ]:
import kagglehub

path = kagglehub.dataset_download('olgaparfenova/daisee')
print(f'Downloaded to: {path}')

DATA_ROOT = '/content/DAiSEE'
shutil.copytree(path, DATA_ROOT, dirs_exist_ok=True)
print('✅ DAISEE ready')

## 🏷️ **5. Load Labels**

In [ ]:
def load_daisee_labels(data_root):
    """
    Load labels from DAISEE CSV files.
    ClipID format in CSV: '110001010101.avi' (includes extension)
    Returns dict: {'110001010101.avi': {engagement, boredom, confusion, frustration, split}}
    """
    label_dict = {}

    for split in ['Train', 'Val', 'Test']:
        candidates = [
            os.path.join(data_root, 'DAiSEE', 'Labels', f'{split}Labels.csv'),
            os.path.join(data_root, 'Labels', f'{split}Labels.csv'),
        ]
        csv_path = next((p for p in candidates if os.path.exists(p)), None)

        if csv_path:
            df = pd.read_csv(csv_path)
            print(f'{split}: {len(df)} clips')
            for _, row in df.iterrows():
                clip_id = str(row['ClipID']).strip()
                label_dict[clip_id] = {
                    'engagement':  int(row['Engagement']),
                    'boredom':     int(row['Boredom']),
                    'confusion':   int(row['Confusion']),
                    'frustration': int(row['Frustration']),
                    'split':       split.lower()
                }
        else:
            print(f'⚠️  No CSV found for {split}')

    print(f'\n✅ Total labels: {len(label_dict)}')
    return label_dict


label_dict = load_daisee_labels(DATA_ROOT)
print(f'Sample key: {list(label_dict.keys())[0]}')

## 🔍 **6. Match Videos to Labels**

In [ ]:
def find_videos_with_labels(data_root, label_dict):
    """
    Build video paths using DAISEE folder structure:
    DataSet/{split}/{subject_id}/{clip_folder}/{clip_id}.avi

    subject_id  = clip_id[:6]              e.g. '110001'
    clip_folder = clip_id without .avi     e.g. '110001010101'
    """
    split_map = {'train': 'Train', 'val': 'Val', 'test': 'Test'}

    dataset_root_candidates = [
        os.path.join(data_root, 'DAiSEE', 'DataSet'),
        os.path.join(data_root, 'DataSet'),
    ]
    dataset_root = next((p for p in dataset_root_candidates if os.path.exists(p)), None)
    print(f'Dataset root: {dataset_root}')

    matched, missing = [], 0

    for clip_id, labels in label_dict.items():
        split_folder = split_map.get(labels['split'], labels['split'].capitalize())
        subject_id   = clip_id[:6]
        clip_folder  = clip_id.replace('.avi', '')

        video_path = os.path.join(dataset_root, split_folder, subject_id, clip_folder, clip_id)

        if os.path.exists(video_path):
            matched.append({
                'video_path':  video_path,
                'clip_name':   clip_folder,
                'engagement':  labels['engagement'],
                'boredom':     labels['boredom'],
                'confusion':   labels['confusion'],
                'frustration': labels['frustration'],
                'split':       labels['split']
            })
        else:
            missing += 1

    splits = {}
    for item in matched:
        splits[item['split']] = splits.get(item['split'], 0) + 1

    print(f'Matched: {len(matched)} | Missing on disk: {missing}')
    print(f'Split distribution: {splits}')
    return matched


all_videos = find_videos_with_labels(DATA_ROOT, label_dict)
print(f'\n✅ Total videos ready: {len(all_videos)}')

## 🧠 **7. AffectNet-ViT Feature Extractor (GPU)**

In [ ]:
class AffectNetViTExtractor:
    """
    ViT pretrained on AffectNet facial expressions.
    Uses GPU batch processing — all frames in one forward pass.
    """

    MODEL_NAME = 'motheecreator/vit-Facial-Expression-Recognition'

    def __init__(self, device='cuda'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        print(f'Loading AffectNet-ViT on {self.device}...')

        self.processor = ViTImageProcessor.from_pretrained(self.MODEL_NAME)
        self.model = ViTModel.from_pretrained(self.MODEL_NAME).to(self.device)

        # Freeze — only use for feature extraction
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

        self.embedding_dim = self.model.config.hidden_size  # 768
        print(f'✅ AffectNet-ViT loaded on {self.device} | Embedding dim: {self.embedding_dim}')

        # Verify GPU
        param_device = next(self.model.parameters()).device
        print(f'Model is on: {param_device}')


extractor = AffectNetViTExtractor(device='cuda')
print(f'\nReady for batch extraction!')

## 🎬 **8. Video Embedding Extraction (Batch + GPU)**

In [ ]:
def extract_video_embeddings(video_path, extractor, max_frames=30, frame_skip=5):
    """
    Extract embeddings from a single video.

    - Reads frames (no YOLO — DAISEE is already face-centered)
    - Sends ALL frames to GPU in ONE batch
    - Returns shape: (max_frames, 768)
    """
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    frame_idx = 0

    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_skip == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb).resize((224, 224))
            frames.append(pil_img)
        frame_idx += 1

    cap.release()

    # Pad with blank frames if video is too short
    while len(frames) < max_frames:
        frames.append(Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)))

    frames = frames[:max_frames]

    # ONE batch forward pass — all 30 frames at once on GPU
    inputs = extractor.processor(images=frames, return_tensors='pt')
    inputs = {k: v.to(extractor.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = extractor.model(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # (30, 768)

    return embeddings.astype(np.float32)


print('✅ extract_video_embeddings defined')

## 🧪 **9. Sanity Check on One Video**

In [ ]:
import time

test_entry = all_videos[0]
print(f'Testing: {test_entry["video_path"]}')

t0 = time.time()
test_emb = extract_video_embeddings(test_entry['video_path'], extractor)
elapsed = time.time() - t0

print(f'\nShape:     {test_emb.shape}')   # Should be (30, 768)
print(f'Dtype:     {test_emb.dtype}')
print(f'Time:      {elapsed:.2f}s')
print(f'Labels:    engagement={test_entry["engagement"]}, boredom={test_entry["boredom"]}, '
      f'confusion={test_entry["confusion"]}, frustration={test_entry["frustration"]}')

assert test_emb.shape == (30, 768), f'Wrong shape: {test_emb.shape}'
print('\n✅ Sanity check passed!')

## 🚀 **10. Full Extraction Pipeline**

In [ ]:
def extract_all_embeddings(video_entries, extractor,
                            save_dir='/content/drive/MyDrive/daisee_embeddings_affectnet',
                            max_frames=30, frame_skip=5):
    """
    Extract and save embeddings for all DAISEE videos.

    Saves:
        {save_dir}/{clip_name}.npy   shape (30, 768)
        {save_dir}/metadata.json     all labels + paths
    """
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    metadata = []
    errors = []

    print(f'Extracting {len(video_entries)} videos...')
    print(f'Saving to: {save_dir}')
    print('='*60)

    for idx, entry in enumerate(tqdm(video_entries, desc='Extracting')):
        clip_name = entry['clip_name']
        emb_path  = save_path / f'{clip_name}.npy'

        # Resume support — skip already extracted
        if emb_path.exists():
            metadata.append({
                'embedding_path': str(emb_path),
                'clip_name':      clip_name,
                'engagement':     entry['engagement'],
                'boredom':        entry['boredom'],
                'confusion':      entry['confusion'],
                'frustration':    entry['frustration'],
                'split':          entry['split']
            })
            continue

        try:
            emb = extract_video_embeddings(
                entry['video_path'], extractor,
                max_frames=max_frames, frame_skip=frame_skip
            )
            np.save(str(emb_path), emb)

            metadata.append({
                'embedding_path': str(emb_path),
                'clip_name':      clip_name,
                'engagement':     entry['engagement'],
                'boredom':        entry['boredom'],
                'confusion':      entry['confusion'],
                'frustration':    entry['frustration'],
                'split':          entry['split']
            })

        except Exception as e:
            print(f'⚠️  Error on {clip_name}: {e}')
            errors.append(clip_name)
            continue

        # Checkpoint every 200 videos
        if (idx + 1) % 200 == 0:
            with open(save_path / 'metadata_checkpoint.json', 'w') as f:
                json.dump(metadata, f)
            print(f'  💾 Checkpoint: {idx+1}/{len(video_entries)} done')

    # Save final metadata
    metadata_path = save_path / 'metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    splits = {}
    for item in metadata:
        splits[item['split']] = splits.get(item['split'], 0) + 1

    print('\n' + '='*60)
    print(f'✅ Done! Extracted: {len(metadata)} | Errors: {len(errors)}')
    print(f'Split distribution: {splits}')
    print(f'Metadata saved: {metadata_path}')

    return metadata


print('✅ extract_all_embeddings defined')

## ▶️ **11. RUN EXTRACTION**

In [ ]:
metadata = extract_all_embeddings(
    video_entries=all_videos,
    extractor=extractor,
    save_dir=SAVE_DIR,
    max_frames=30,
    frame_skip=5
)

## ✅ **12. Verify Results**

In [ ]:
import random
import matplotlib.pyplot as plt

# Load metadata
with open(f'{SAVE_DIR}/metadata.json') as f:
    metadata = json.load(f)

print(f'Total clips: {len(metadata)}')

# Verify shapes on 10 random samples
samples = random.sample(metadata, 10)
all_ok = True
for s in samples:
    emb = np.load(s['embedding_path'])
    if emb.shape != (30, 768):
        print(f'❌ Wrong shape: {s["clip_name"]} -> {emb.shape}')
        all_ok = False
if all_ok:
    print('✅ All shapes correct: (30, 768)')

# Label distribution plot
df = pd.DataFrame(metadata)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, emotion in zip(axes, ['engagement', 'boredom', 'confusion', 'frustration']):
    counts = df[emotion].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='black')
    ax.set_title(emotion.capitalize(), fontsize=13)
    ax.set_xlabel('Level (0–3)')
    ax.set_ylabel('Count')
    for i, v in zip(counts.index, counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontsize=9)
plt.suptitle('DAISEE Label Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/label_distribution.png', dpi=100)
plt.show()

print('\n✅ Stage A complete! Ready for Stage B.')

## 📋 **13. How to Load in Stage B**

```python
import json
from sklearn.model_selection import train_test_split

EMBEDDING_DIR = '/content/drive/MyDrive/daisee_embeddings_affectnet'

with open(f'{EMBEDDING_DIR}/metadata.json') as f:
    metadata = json.load(f)

train_meta = [m for m in metadata if m['split'] == 'train']
test_meta  = [m for m in metadata if m['split'] == 'test']

# Make val split from train (80/20)
train_meta, val_meta = train_test_split(train_meta, test_size=0.2, random_state=42)

print(f'Train: {len(train_meta)} | Val: {len(val_meta)} | Test: {len(test_meta)}')
```

Embedding shape: `(30, 768)` — no changes needed to LSTM/KAN/CORAL.